# TT-14 — ElasticNet: Dự báo tải sưởi (Y1) và tải làm mát (Y2)
### Bộ dữ liệu Energy Efficiency (UCI) — 768 dòng × 8 đặc trưng

**Mục tiêu:** Chứng minh ElasticNet xử lý tốt đa cộng tuyến (X1, X2, X4, X5 tương quan gần hoàn hảo)
tốt hơn Lasso, đồng thời vẫn loại được biến vô ích tốt hơn Ridge.

Notebook đi theo đúng 10 bước trong README. Mỗi cell nên chạy tuần tự, không skip.


## 0. Import thư viện & thiết lập

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
)
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)


## Bước 1 — Nạp dữ liệu, tính ma trận tương quan + VIF
Xác nhận đa cộng tuyến nặng giữa X1, X2, X4, X5 (đúng như README cảnh báo).


In [ ]:
# ⚠️ ĐƯỜNG DẪN DỮ LIỆU — chỉnh DATA_PATH nếu cấu trúc thư mục của bạn khác.
# Mặc định: notebook và file .xlsx nằm CÙNG thư mục (khuyến nghị khi nộp bài,
# vì tránh việc hardcode path riêng của một máy khiến notebook "chết" trên máy khác).
import os

DATA_PATH = "ENB2012_data.xlsx"

# Nếu không thấy ở thư mục hiện tại, thử tìm trong vài vị trí phổ biến theo
# cấu trúc thư mục gợi ý ở README (TT-14/energy+efficiency/...)
_candidates = [
    DATA_PATH,
    os.path.join("energy+efficiency", "ENB2012_data.xlsx"),
    os.path.join("data", "ENB2012_data.xlsx"),
]
_found = next((p for p in _candidates if os.path.exists(p)), None)

if _found is None:
    raise FileNotFoundError(
        "Không tìm thấy file dữ liệu ENB2012_data.xlsx.\n"
        f"Đã thử các đường dẫn: {_candidates}\n"
        "-> Sửa lại biến DATA_PATH ở trên cho đúng vị trí file trên máy bạn, "
        "ví dụ: DATA_PATH = r\"D:\\duong\\dan\\that\\ENB2012_data.xlsx\""
    )

DATA_PATH = _found
print("Đang đọc dữ liệu từ:", os.path.abspath(DATA_PATH))
df = pd.read_excel(DATA_PATH)

# Đổi tên cột cho dễ đọc (giữ nguyên X1..X8, Y1, Y2 để khớp README)
col_meaning = {
    "X1": "Relative_Compactness",
    "X2": "Surface_Area",
    "X3": "Wall_Area",
    "X4": "Roof_Area",
    "X5": "Overall_Height",
    "X6": "Orientation",          # phân loại
    "X7": "Glazing_Area",
    "X8": "Glazing_Area_Dist",    # phân loại
    "Y1": "Heating_Load",
    "Y2": "Cooling_Load",
}

print("Kích thước dữ liệu:", df.shape)
print("\nKiểu dữ liệu:")
print(df.dtypes)
print("\nGiá trị thiếu (nếu có):")
print(df.isnull().sum())

df.head()


In [ ]:
# Ma trận tương quan (chỉ biến số liên tục, X6/X8 sẽ xử lý one-hot ở Bước 2)
numeric_cols = ["X1", "X2", "X3", "X4", "X5", "X7"]
corr = df[numeric_cols + ["Y1", "Y2"]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Ma trận tương quan giữa các biến")
plt.tight_layout()
plt.savefig("reports/correlation_heatmap.png", dpi=150)
plt.show()

print("\n⚠️ Kiểm tra: |r| > 0.95 giữa X1, X2, X4, X5?")
pairs_high = []
for i, a in enumerate(numeric_cols):
    for b in numeric_cols[i+1:]:
        r = df[a].corr(df[b])
        if abs(r) > 0.95:
            pairs_high.append((a, b, round(r, 3)))
print(pairs_high)


In [ ]:
# VIF (Variance Inflation Factor) — đo đa cộng tuyến trực tiếp
# VIF > 10 (nhiều tài liệu dùng ngưỡng 5 hoặc 10) => đa cộng tuyến nghiêm trọng
X_vif = df[numeric_cols].copy()
X_vif = (X_vif - X_vif.mean()) / X_vif.std()  # chuẩn hoá trước khi tính VIF cho ổn định số học
X_vif = X_vif.assign(const=1)  # statsmodels cần cột hằng số

vif_data = pd.DataFrame()
vif_data["feature"] = numeric_cols
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i) for i in range(len(numeric_cols))
]
vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)
vif_data.to_csv("reports/vif_table.csv", index=False)

print(vif_data)
print("\n👉 Biến nào VIF cao chứng tỏ đa cộng tuyến nặng, đúng như README cảnh báo về nhóm X1-X2-X4-X5.")


## Bước 2 — One-hot encode X6, X8; chuẩn hoá các biến số
X6 (hướng nhà) và X8 (phân bố kính) là biến **phân loại** dù mã hoá bằng số →
phải one-hot, tuyệt đối không để dạng số có thứ tự (kẻo model hiểu nhầm "hướng 4 > hướng 2").


In [ ]:
numeric_features = ["X1", "X2", "X3", "X4", "X5", "X7"]
categorical_features = ["X6", "X8"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features),
    ]
)

X = df[numeric_features + categorical_features]
y1 = df["Y1"]  # tải sưởi
y2 = df["Y2"]  # tải làm mát

print("Số cột sau one-hot (ước tính):", len(numeric_features) +
      (df["X6"].nunique() - 1) + (df["X8"].nunique() - 1))


## Bước 3 — Baseline: DummyRegressor + Linear Regression
Đây là mốc để so sánh: nếu ElasticNet không tốt hơn baseline rõ rệt thì có vấn đề.


In [ ]:
def evaluate_baseline(X, y, target_name, random_state=RANDOM_STATE):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    results = {}
    for name, model in [
        ("Dummy (mean)", DummyRegressor(strategy="mean")),
        ("Linear Regression", LinearRegression()),
    ]:
        pipe = Pipeline([("prep", preprocessor), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)
        results[name] = {"RMSE": rmse, "R2": r2}

    return pd.DataFrame(results).T, (X_train, X_test, y_train, y_test)

baseline_Y1, split_Y1 = evaluate_baseline(X, y1, "Y1")
baseline_Y2, split_Y2 = evaluate_baseline(X, y2, "Y2")

print("=== Baseline Y1 (Heating Load) ===")
print(baseline_Y1)
print("\n=== Baseline Y2 (Cooling Load) ===")
print(baseline_Y2)


## Bước 4 — Chạy 3 model trên CÙNG dữ liệu: Ridge · Lasso · ElasticNet (cho Y1 trước)
Dùng `*CV` để tự dò `alpha` bằng cross-validation 5-fold (đủ ổn định với 768 dòng, đúng khuyến cáo README).
`ElasticNetCV` còn dò thêm `l1_ratio` — chính là điểm khác biệt cốt lõi.


In [ ]:
def fit_regularized_models(X_train, y_train, l1_ratios=(0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0)):
    alphas = np.logspace(-4, 1, 100)

    models = {
        "Ridge": Pipeline([
            ("prep", preprocessor),
            ("model", RidgeCV(alphas=alphas, cv=5)),
        ]),
        "Lasso": Pipeline([
            ("prep", preprocessor),
            ("model", LassoCV(alphas=alphas, cv=5, max_iter=50000, random_state=RANDOM_STATE)),
        ]),
        "ElasticNet": Pipeline([
            ("prep", preprocessor),
            ("model", ElasticNetCV(
                l1_ratio=list(l1_ratios),
                alphas=alphas,
                cv=5, max_iter=50000, random_state=RANDOM_STATE)),
        ]),
    }

    for name, pipe in models.items():
        pipe.fit(X_train, y_train)

    return models

X_train_Y1, X_test_Y1, y_train_Y1, y_test_Y1 = split_Y1
models_Y1 = fit_regularized_models(X_train_Y1, y_train_Y1)

print("alpha tối ưu:")
print("  Ridge      :", models_Y1["Ridge"]["model"].alpha_)
print("  Lasso      :", models_Y1["Lasso"]["model"].alpha_)
print("  ElasticNet :", models_Y1["ElasticNet"]["model"].alpha_,
      "| l1_ratio:", models_Y1["ElasticNet"]["model"].l1_ratio_)


## Bước 5 — ⭐ Bảng so sánh: mỗi model giữ bao nhiêu biến? RMSE bao nhiêu?


In [ ]:
def get_feature_names(preprocessor):
    num_names = numeric_features
    cat_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features))
    return num_names + cat_names

def compare_models(models, X_test, y_test, target_name):
    rows = []
    for name, pipe in models.items():
        pred = pipe.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)
        coefs = pipe["model"].coef_
        n_nonzero = int(np.sum(np.abs(coefs) > 1e-6))
        rows.append({
            "Model": name,
            "RMSE": round(rmse, 4),
            "R2": round(r2, 4),
            "So_bien_giu_lai": n_nonzero,
            "Tong_so_bien": len(coefs),
        })
    result = pd.DataFrame(rows).set_index("Model")
    print(f"=== So sánh 3 model cho {target_name} ===")
    print(result)
    return result

feature_names = get_feature_names(models_Y1["ElasticNet"]["prep"])
compare_Y1 = compare_models(models_Y1, X_test_Y1, y_test_Y1, "Y1 (Heating Load)")
compare_Y1.to_csv("reports/so_sanh_3_model_Y1.csv")


In [ ]:
# Biểu đồ so sánh RMSE + số biến giữ lại
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

compare_Y1["RMSE"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("RMSE theo model (Y1)")
axes[0].set_ylabel("RMSE")

compare_Y1["So_bien_giu_lai"].plot(kind="bar", ax=axes[1], color="#DD8452")
axes[1].set_title("Số biến giữ lại (Y1)")
axes[1].set_ylabel("Số biến (khác 0)")

plt.tight_layout()
plt.savefig("reports/so_sanh_3_model.png", dpi=150)
plt.show()


## Bước 6 — ⭐ Kiểm chứng HIỆU ỨNG GOM NHÓM
Nhóm biến dính chặt nhau: **X1 (độ gọn), X2 (diện tích bề mặt), X4 (diện tích mái), X5 (chiều cao)**.
- Lasso kỳ vọng: chọn 1 biến trong nhóm, hệ số các biến còn lại ≈ 0.
- ElasticNet kỳ vọng: giữ lại nhiều hơn 1 biến trong nhóm (grouping effect).


In [ ]:
group_vars = ["X1", "X2", "X4", "X5"]

coef_table = pd.DataFrame({
    name: pipe["model"].coef_ for name, pipe in models_Y1.items()
}, index=feature_names)

print("=== Hệ số hồi quy (đã chuẩn hoá) cho nhóm biến tương quan ===")
print(coef_table.loc[group_vars].round(4))

print("\n=== Toàn bộ hệ số (để tham khảo) ===")
print(coef_table.round(4))

print("\n👉 Nhận xét cần điền vào README phần 'HIỆU ỨNG GOM NHÓM':")
for var in group_vars:
    lasso_c = coef_table.loc[var, "Lasso"]
    en_c = coef_table.loc[var, "ElasticNet"]
    print(f"  {var}: Lasso={lasso_c:.4f}  |  ElasticNet={en_c:.4f}"
          f"  {'-> Lasso đã LOẠI biến này' if abs(lasso_c) < 1e-6 else ''}")


## Bước 7 — Vẽ heatmap RMSE theo lưới (alpha × l1_ratio)

**Vì sao cần cả `ElasticNetCV` (Bước 4) LẪN `GridSearchCV` thủ công (cell dưới)?**

| | `ElasticNetCV` | `GridSearchCV` |
|---|---|---|
| Mục đích | **Chốt** alpha/l1_ratio tốt nhất để dùng cho model cuối | **Trực quan hoá** toàn bộ bề mặt RMSE trên lưới |
| Cách chạy | Với mỗi l1_ratio, dùng coordinate descent dò theo *đường dẫn alpha* (warm start) → rất nhanh | Chạy CV độc lập cho từng cặp (alpha, l1_ratio) trong lưới → chậm hơn nhưng cho điểm số ở MỌI ô |
| Kết quả trả về | 1 cặp (alpha, l1_ratio) tối ưu duy nhất | Ma trận RMSE đầy đủ 15×7 → vẽ được heatmap |
| Dùng để | Huấn luyện model chính thức (Bước 4, 8) | Chỉ để minh hoạ hình dạng bề mặt loss, xem "sát mép ngưỡng" tối ưu ra sao |

Nói cách khác: `ElasticNetCV` là công cụ *sản xuất*, `GridSearchCV` ở đây là công cụ *quan sát*.
Không thể lấy ma trận điểm số đầy đủ từ `ElasticNetCV` (nó không lưu lại full grid), nên cần
chạy `GridSearchCV` riêng chỉ để phục vụ heatmap — không dùng kết quả này để chọn model cuối
(model cuối vẫn lấy từ `ElasticNetCV` ở Bước 4/8 để tránh chạy CV hai lần cho cùng một việc).


In [ ]:
alpha_grid = np.logspace(-3, 1, 15)
l1_ratio_grid = np.array([0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0])

param_grid = {
    "model__alpha": alpha_grid,
    "model__l1_ratio": l1_ratio_grid,
}

en_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", ElasticNet(max_iter=50000, random_state=RANDOM_STATE)),
])

grid = GridSearchCV(
    en_pipe, param_grid, cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
grid.fit(X_train_Y1, y_train_Y1)

results_df = pd.DataFrame(grid.cv_results_)
rmse_pivot = results_df.pivot_table(
    index="param_model__l1_ratio",
    columns="param_model__alpha",
    values="mean_test_score",
) * -1  # chuyển neg RMSE -> RMSE dương

plt.figure(figsize=(11, 5))
sns.heatmap(rmse_pivot, cmap="viridis_r", annot=False,
            xticklabels=[f"{a:.3f}" for a in alpha_grid])
plt.title("CV RMSE theo lưới alpha × l1_ratio (Y1)")
plt.xlabel("alpha")
plt.ylabel("l1_ratio")
plt.tight_layout()
plt.savefig("reports/heatmap_alpha_l1ratio.png", dpi=150)
plt.show()

print("Điểm tốt nhất trên lưới:", grid.best_params_, "| RMSE:", -grid.best_score_)


## Bước 8 — Làm cả hai nhãn Y1 và Y2 → so sánh biến quan trọng


In [ ]:
X_train_Y2, X_test_Y2, y_train_Y2, y_test_Y2 = split_Y2
models_Y2 = fit_regularized_models(X_train_Y2, y_train_Y2)

print("alpha / l1_ratio tối ưu (Y2):")
print("  ElasticNet :", models_Y2["ElasticNet"]["model"].alpha_,
      "| l1_ratio:", models_Y2["ElasticNet"]["model"].l1_ratio_)

compare_Y2 = compare_models(models_Y2, X_test_Y2, y_test_Y2, "Y2 (Cooling Load)")
compare_Y2.to_csv("reports/so_sanh_3_model_Y2.csv")


In [ ]:
# So sánh hệ số ElasticNet giữa Y1 và Y2 -> biến nào quan trọng cho sưởi vs làm mát
coef_compare = pd.DataFrame({
    "ElasticNet_Y1": models_Y1["ElasticNet"]["model"].coef_,
    "ElasticNet_Y2": models_Y2["ElasticNet"]["model"].coef_,
}, index=feature_names)

coef_compare["Chenh_lech"] = (coef_compare["ElasticNet_Y1"] - coef_compare["ElasticNet_Y2"]).abs()
coef_compare = coef_compare.sort_values("Chenh_lech", ascending=False)

print(coef_compare.round(4))

coef_compare[["ElasticNet_Y1", "ElasticNet_Y2"]].plot(kind="bar", figsize=(10, 5))
plt.title("So sánh hệ số ElasticNet: Y1 (sưởi) vs Y2 (làm mát)")
plt.ylabel("Hệ số (đã chuẩn hoá)")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("reports/coef_compare_Y1_Y2.png", dpi=150)
plt.show()


## Bước 9 — Kiểm tra ổn định: bootstrap 100 lần

Hệ số ElasticNet dao động bao nhiêu qua 100 lần resample **có hoàn lại (with replacement)**?

⚠️ Bootstrap phải chạy trên **tập train** (`X_train_Y1`, `y_train_Y1`), không phải toàn bộ
`X, y1` — nếu resample trên cả tập test thì test set không còn "chưa từng thấy" nữa, và mọi
đánh giá RMSE/R2 ở các bước trước sẽ mất ý nghĩa (data leakage). Mục tiêu ở đây chỉ là đo độ
ổn định của hệ số khi dữ liệu huấn luyện thay đổi nhẹ, nên chỉ resample trong train.


In [ ]:
def bootstrap_coefficients(X_train, y_train, best_alpha, best_l1_ratio, n_boot=100, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = len(X_train)
    coef_list = []

    # reset_index để rng.choice(idx) map đúng vị trí .iloc, tránh lệch do index gốc không liên tục
    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    for i in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)  # có hoàn lại (with replacement)
        X_boot = X_train.iloc[idx]
        y_boot = y_train.iloc[idx]

        pipe = Pipeline([
            ("prep", preprocessor),
            ("model", ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio,
                                  max_iter=50000, random_state=random_state)),
        ])
        pipe.fit(X_boot, y_boot)
        coef_list.append(pipe["model"].coef_)

    return np.array(coef_list)

best_alpha_Y1 = models_Y1["ElasticNet"]["model"].alpha_
best_l1_Y1 = models_Y1["ElasticNet"]["model"].l1_ratio_

# CHỈ resample trên tập train (X_train_Y1, y_train_Y1) — không dùng toàn bộ X, y1
boot_coefs = bootstrap_coefficients(X_train_Y1, y_train_Y1, best_alpha_Y1, best_l1_Y1, n_boot=100)

boot_summary = pd.DataFrame({
    "feature": feature_names,
    "mean_coef": boot_coefs.mean(axis=0),
    "std_coef": boot_coefs.std(axis=0),
    "cv_percent": np.abs(boot_coefs.std(axis=0) / (boot_coefs.mean(axis=0) + 1e-9)) * 100,
}).sort_values("std_coef", ascending=False)

print(boot_summary.round(4))
boot_summary.to_csv("reports/bootstrap_stability.csv", index=False)


In [ ]:
# Boxplot phân phối hệ số qua 100 lần bootstrap
plt.figure(figsize=(11, 5))
plt.boxplot(boot_coefs, labels=feature_names, showfliers=False)
plt.xticks(rotation=45, ha="right")
plt.axhline(0, color="red", linestyle="--", linewidth=0.8)
plt.title("Phân phối hệ số ElasticNet qua 100 lần bootstrap (Y1)")
plt.tight_layout()
plt.savefig("reports/bootstrap_boxplot.png", dpi=150)
plt.show()


## Lưu model đã huấn luyện

In [ ]:
joblib.dump(models_Y1["ElasticNet"], "models/elasticnet_Y1.joblib")
joblib.dump(models_Y2["ElasticNet"], "models/elasticnet_Y2.joblib")
print("Đã lưu models/elasticnet_Y1.joblib và models/elasticnet_Y2.joblib")


## Bước 10 — ✍️ Đề xuất thay đổi thiết kế

Cell dưới in ra top biến ảnh hưởng nhất theo hệ số ElasticNet (đã chuẩn hoá) cho từng nhãn,
để bạn **đối chiếu số liệu thực tế sau khi chạy** với 3 đề xuất thiết kế dưới đây.

Ba đề xuất này dựa trên bản chất vật lý của bài toán (không phải chỉ đoán mò), và cũng khớp
với các phát hiện đã công bố trên chính bộ dữ liệu ENB2012 (Tsanas & Xifara, 2012):

**1. Ưu tiên tăng độ gọn hình học (Relative Compactness — X1), không chỉ giảm diện tích sàn.**
   X1 = tỉ lệ thể tích / diện tích bề mặt bao che. Toà nhà càng "gọn" (gần khối lập phương/cầu)
   thì diện tích bề mặt tiếp xúc với môi trường trên mỗi m³ không gian sử dụng càng nhỏ →
   giảm cả tải sưởi lẫn tải làm mát. Đây thường là biến có hệ số lớn và ổn định nhất (bootstrap
   dao động thấp) vì nó "gói gọn" ảnh hưởng của cả nhóm X2/X4/X5 — chính là lý do ElasticNet
   giữ lại cả nhóm thay vì chỉ 1 biến như Lasso.

**2. Kiểm soát chiều cao tầng (Overall Height — X5) khi thiết kế, đặc biệt với tải sưởi (Y1).**
   Chiều cao lớn hơn thường làm tăng diện tích tường bao (X3) tương ứng, tăng thất thoát nhiệt
   mùa đông. Nếu hệ số ElasticNet của X5 cho Y1 dương và lớn hơn rõ rệt so với Y2, đây là bằng
   chứng cho thấy nên ưu tiên tối ưu chiều cao trước khi tối ưu các yếu tố khác nếu mục tiêu là
   giảm chi phí sưởi ấm.

**3. Giảm diện tích kính (Glazing Area — X7) hoặc dùng kính cách nhiệt, đặc biệt với tải làm mát (Y2).**
   Kính có hệ số cách nhiệt kém hơn tường đặc nhiều lần, và hấp thụ bức xạ mặt trời trực tiếp →
   thường là biến có ảnh hưởng RÕ RỆT hơn tới Y2 (làm mát) so với Y1 (sưởi) trong dữ liệu này.
   Nếu bảng so sánh hệ số Y1 vs Y2 ở Bước 8 cho thấy |hệ số X7 với Y2| > |hệ số X7 với Y1| đáng
   kể, đó là cơ sở để đề xuất giảm diện tích kính hướng nắng gắt hoặc dùng kính Low-E thay vì
   chỉ giảm đều diện tích kính ở mọi hướng.

> Lưu ý: dấu và độ lớn hệ số cụ thể phụ thuộc vào lần chạy thực tế (do `random_state` cố định
> nên kết quả sẽ tái lập được, nhưng bạn vẫn nên chạy cell dưới để xác nhận trước khi viết vào
> báo cáo, thay vì chỉ dùng lại các con số trong tài liệu gốc).


In [ ]:
def top_features(pipe, feature_names, target_name, top_n=5):
    coefs = pipe["model"].coef_
    s = pd.Series(coefs, index=feature_names).sort_values(key=np.abs, ascending=False)
    print(f"--- Top {top_n} bien anh huong nhat toi {target_name} ---")
    print(s.head(top_n).round(4))
    print()
    return s

top_Y1 = top_features(models_Y1["ElasticNet"], feature_names, "Y1 (Heating Load)")
top_Y2 = top_features(models_Y2["ElasticNet"], feature_names, "Y2 (Cooling Load)")

# Doi chieu nhanh voi 3 de xuat o markdown phia tren
x1_y1 = models_Y1["ElasticNet"]["model"].coef_[feature_names.index("X1")]
x1_y2 = models_Y2["ElasticNet"]["model"].coef_[feature_names.index("X1")]
print("Kiem chung de xuat #1 (X1 on dinh, he so lon):")
print(f"  X1 -> Y1: {x1_y1:.4f} | X1 -> Y2: {x1_y2:.4f}")

x5_y1 = models_Y1["ElasticNet"]["model"].coef_[feature_names.index("X5")]
x5_y2 = models_Y2["ElasticNet"]["model"].coef_[feature_names.index("X5")]
verdict_2 = "DUNG nhu ky vong" if abs(x5_y1) > abs(x5_y2) else "NGUOC voi ky vong, can xem lai de xuat #2"
print(f"\nKiem chung de xuat #2 (X5 anh huong Y1 > Y2):")
print(f"  X5 -> Y1: {x5_y1:.4f} | X5 -> Y2: {x5_y2:.4f} | {verdict_2}")

x7_y1 = models_Y1["ElasticNet"]["model"].coef_[feature_names.index("X7")]
x7_y2 = models_Y2["ElasticNet"]["model"].coef_[feature_names.index("X7")]
verdict_3 = "DUNG nhu ky vong" if abs(x7_y2) > abs(x7_y1) else "NGUOC voi ky vong, can xem lai de xuat #3"
print(f"\nKiem chung de xuat #3 (X7 anh huong Y2 > Y1):")
print(f"  X7 -> Y1: {x7_y1:.4f} | X7 -> Y2: {x7_y2:.4f} | {verdict_3}")


## Tổng kết

Đối chiếu với tiêu chí hoàn thành trong README:
- [x] Bảng VIF chứng minh đa cộng tuyến (`reports/vif_table.csv`) — Bước 1
- [x] Bảng so sánh Ridge / Lasso / ElasticNet — số biến giữ + RMSE (`reports/so_sanh_3_model_Y1.csv`, `_Y2.csv`) — Bước 5, 8
- [x] Hiệu ứng gom nhóm: bảng hệ số nhóm X1/X2/X4/X5 — Bước 6
- [x] Heatmap alpha × l1_ratio (`reports/heatmap_alpha_l1ratio.png`) — Bước 7
- [x] Làm đủ cả 2 nhãn Y1 và Y2 bằng 2 model riêng biệt, có so sánh hệ số — Bước 8
- [x] Ý nghĩa l1_ratio máy chọn — in ra ở Bước 4/8 (`l1_ratio_` gần 1 → giống Lasso, gần 0 → giống Ridge)
- [x] 3 đề xuất thiết kế cụ thể, có kiểm chứng bằng số liệu thực — Bước 10
- [ ] **Cần bạn tự đối chiếu sau khi chạy:** so RMSE của bảng Bước 5/Bước 8 với baseline ở Bước 3 —
      nếu ElasticNet không tốt hơn Linear Regression rõ rệt, khả năng cao đây không phải bài toán
      cần regularization mạnh (ít khả năng xảy ra với dữ liệu mô phỏng sạch như ENB2012, nhưng vẫn nên xác nhận).

**Mức tham chiếu README:** R² ~0,90–0,92 cho Y1 (tải sưởi). So kết quả bảng `compare_Y1` ở Bước 5
với mức này — nếu thấp hơn nhiều, kiểm tra lại bước one-hot/chuẩn hoá hoặc `max_iter` có đủ để hội tụ không.
